# Export a saved LoRA adapter to GGUF (no training)

Use this when a training run finished and saved `lora_adapter/`, but the GGUF export
did not complete (for example, the run hit Kaggle's 12-hour limit). Takes ~30–40 minutes.

**Settings:** GPU T4, Internet On. **Add Input** → the training run's output (the version
whose Output tab shows `lora_adapter`). Then **Save & Run All (Commit)** and download the
`.gguf` from this version's Output tab.

In [ ]:
!pip install -q unsloth

In [ ]:
import glob, os
# Any folder with adapter_config.json: an attached training output (lora_adapter/) or an
# uploaded dataset (files at its root). Prefer the final adapter over training checkpoints.
hits = sorted(glob.glob("/kaggle/input/**/adapter_config.json", recursive=True))
hits.sort(key=lambda h: ("checkpoint-" in h, "lora_adapter" not in h))
assert hits, "adapter not found: Add Input -> the upload/output that contains adapter_config.json"
ADAPTER = os.path.dirname(hits[0])
OUT_GGUF = "/tmp/gguf_model"   # big export files live outside /kaggle/working (~20 GB output limit)
print("adapter:", ADAPTER)
print(sorted(os.listdir(ADAPTER)))

In [ ]:
# Loading the adapter folder loads its base model (Foundation-Sec-8B-Instruct) plus the trained LoRA.
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=ADAPTER, max_seq_length=2048, load_in_4bit=True, dtype=None,
)

In [ ]:
# Same chat template as training, so the exported model matches make_modelfile.py
from huggingface_hub import hf_hub_download
tokenizer.chat_template = open(hf_hub_download("fdtn-ai/Foundation-Sec-8B-Instruct", "chat_template.jinja")).read()
assert tokenizer.chat_template.startswith("{{ bos_token }}"), "unexpected chat template"

In [ ]:
!df -h /kaggle/working /tmp
# Merges the LoRA into 16-bit weights, converts to GGUF, quantizes to Q4_K_M (~20 min)
model.save_pretrained_gguf(OUT_GGUF, tokenizer, quantization_method="q4_k_m")

In [ ]:
import glob, os, shutil
ggufs = sorted(glob.glob("/tmp/**/*.gguf", recursive=True))
print("GGUF files:", ggufs)
q4 = [g for g in ggufs if "q4_k_m" in g.lower()] or ggufs
assert q4, "no .gguf produced: check the export log above"
shutil.copy(q4[0], "/kaggle/working/")
print("saved to output:", os.path.basename(q4[0]))
# Download it from this version's Output tab. Then on your own machine, in the cyber-llm repo:
#   python3 make_modelfile.py ~/Downloads/<file>.gguf
#   ollama create cyber-sec -f ~/Downloads/Modelfile && ollama run cyber-sec